**Legacy notebook.** Self-contained analysis code that predates the `src/mrvf` library and has not been ported to it. Kept for provenance and because it still produces figures in `results/`. Paths were updated to the `results/` layout; the next cell sets the working directory to the repository root, so run it from anywhere.

For the maintained pipeline see `notebooks/01_train_triple_regime.ipynb` and `notebooks/02_evaluate_rmse_vs_snr.ipynb`.

In [ ]:
import os
from pathlib import Path
# run from the repository root so ./results/... and ../subsamples resolve
_root = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src" / "mrvf").is_dir())
os.chdir(_root)

# Ablation: RMSE vs SNR — DM vs DL-4p vs Triple-NF vs Triple-Noisy

All four methods are evaluated on the **same shared test set** at each SNR level.

| Method | Input dim | Training noise | Checkpoint |
|--------|-----------|----------------|------------|
| DM | 40 (L2-norm) | — | noise-free dictionary |
| DL-4p | 40 (L2-norm) | mixed-SNR | `t2snr_results_v4/models/t2snr_noisy_4param_v4.pt` |
| Triple-NF | 43 (signal + feat_A/B/C) | noise-free | `triple_regime_nf_results_v1/models/triple_nf_best.pt` |
| Triple-Noisy | 43 (signal + feat_A/B/C) | mixed-SNR | `triple_regime_results_v1/models/triple_regime_best.pt` |

Data: `../subsamples/subsamples_v3/`  
Parameters: SO₂ (%), CBV (%), R (µm), T2 (ms)  
SNR levels: 20, 50, 100, 150

## 1. Imports & GPU

In [ ]:
import os, json, time
import numpy as np
import scipy.io as sio
import h5py
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn

plt.rcParams.update({
    'font.family'    : 'Arial',
    'font.size'      : 9,
    'axes.labelsize' : 10,
    'axes.titlesize' : 11,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.dpi'     : 150,
    'savefig.dpi'    : 300,
    'savefig.bbox'   : 'tight',
})

C_DM          = '#E65100'   # deep orange
C_DL_4P       = '#1565C0'   # deep blue
C_TRIPLE_NF   = '#AD1457'   # deep pink
C_TRIPLE_NOISY= '#6A1B9A'   # deep purple

if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    device = torch.device('cpu')
    print('CPU only')
print(f'PyTorch {torch.__version__}')

## 2. Configuration

In [ ]:
CONFIG = {
    # ── Data paths ────────────────────────────────────────────────────────────
    'dict_base_path'    : '../subsamples/subsamples_v3',
    'param_path'        : '../subsamples/subsamples_v3/QuasiRand_par_t2_200.mat',
    'noisefree_sig_path': '../subsamples/subsamples_v3/QuasiRand_t2_200.mat',
    'echotimes_path'    : '../echotimes.mat',
    'dict_key'          : 'Dico40_save',
    'param_key'         : 'par_save',

    # ── Checkpoint paths ──────────────────────────────────────────────────────
    'ckpt_dl4p'         : './results/t2snr_results_v4/models/t2snr_noisy_4param_v4.pt',
    'ckpt_triple_nf'    : './results/triple_regime_nf_results_v1/models/triple_nf_best.pt',
    'ckpt_triple_noisy' : './results/triple_regime_results_v1/models/triple_regime_best.pt',

    # ── Parameter space ───────────────────────────────────────────────────────
    'param_mins'  : np.array([0.0,    0.0025,  1.0e-6,  0.050]),
    'param_maxs'  : np.array([1.0,    0.15,   25.0e-6,  0.200]),
    'param_names' : ['SO2', 'CBV', 'R', 'T2'],

    # ── GESFIDE geometry ──────────────────────────────────────────────────────
    'n_fid'   : 14,
    'n_rephas': 16,
    'n_postse': 10,

    # ── Feature scaling for Triple regime ────────────────────────────────────
    'R2starA_min':  2.0,   'R2starA_max': 55.0,
    'R2starB_min': -30.0,  'R2starB_max': 22.0,
    'R2starC_min':  2.0,   'R2starC_max': 55.0,

    # ── Evaluation ───────────────────────────────────────────────────────────
    'snr_levels'    : [20, 50, 100, 150],
    'test_frac'     : 0.15,   # must match original training splits
    'seed'          : 42,
    'dl_batch_size' : 4096,
    'dm_batch_size' : 256,    # smaller to avoid GPU OOM

    'output_dir': './results/ablation_results',
}

SE_ECHO = CONFIG['n_fid'] + CONFIG['n_rephas']   # = 30

PARAM_NAMES = ['SO₂', 'CBV', 'R',   'T2']
PARAM_UNITS = ['(%)', '(%)', '(µm)', '(ms)']
PARAM_SCALE = [100,   100,   1e6,    1000]
PKEYS       = CONFIG['param_names']
SNR_LEVELS  = CONFIG['snr_levels']

os.makedirs(CONFIG['output_dir'], exist_ok=True)
FIG_DIR = os.path.join(CONFIG['output_dir'], 'figures')
os.makedirs(FIG_DIR, exist_ok=True)
print(f'SE_ECHO = {SE_ECHO}')
print(f'Output : {CONFIG["output_dir"]}')

## 3. Utility functions

In [ ]:
def load_mat(path, key):
    try:
        mat = sio.loadmat(path)
        if key in mat:
            return np.array(mat[key], dtype=np.float32)
        cands = [k for k in mat if not k.startswith('_')]
        print(f'  Key "{key}" not found, using "{cands[0]}"')
        return np.array(mat[cands[0]], dtype=np.float32)
    except NotImplementedError:
        with h5py.File(path, 'r') as f:
            data = f[key][()] if key in f else f[next(k for k in f if not k.startswith('#'))][()]
            if data.ndim >= 2: data = data.T
            return np.array(data, dtype=np.float32)

def euclidean_norm(data):
    data = np.abs(data).astype(np.float32)
    return data / np.maximum(np.linalg.norm(data, axis=1, keepdims=True), 1e-12)

def params_scale(p, mins, maxs):
    return ((p - mins) / (maxs - mins)).astype(np.float32)

def params_inverse(p, mins, maxs):
    return (p * (maxs - mins) + mins).astype(np.float32)

def filter_param_range(signals, params, mins, maxs):
    mask = np.ones(len(params), dtype=bool)
    for i in range(min(params.shape[1], len(mins))):
        mask &= (params[:, i] >= mins[i]) & (params[:, i] <= maxs[i])
    if (~mask).sum():
        print(f'  Filtered {(~mask).sum()} out-of-range ({(~mask).mean()*100:.1f}%)')
    return signals[mask], params[mask]

def clean_data(signals, params):
    valid = np.all(np.isfinite(signals), axis=1) & np.all(np.isfinite(params), axis=1)
    if (~valid).sum(): print(f'  Removed {(~valid).sum()} non-finite entries')
    return signals[valid], params[valid]

def batched_predict(model, x_np, batch_size=4096):
    model.eval(); preds = []
    with torch.no_grad():
        for i in range(0, len(x_np), batch_size):
            xb = torch.tensor(x_np[i:i+batch_size], dtype=torch.float32).to(device).contiguous()
            preds.append(model(xb).cpu().numpy())
    return np.concatenate(preds, axis=0)

def compute_rmse(pred_raw, true_raw):
    """Returns dict of RMSE per parameter in physical units."""
    rmse = {}
    for i, (name, sc) in enumerate(zip(PKEYS, PARAM_SCALE)):
        rmse[name] = float(np.sqrt(np.mean((pred_raw[:, i]*sc - true_raw[:, i]*sc)**2)))
    return rmse

def save_fig(fig, name):
    for ext in ['png', 'pdf']:
        p = os.path.join(FIG_DIR, f'{name}.{ext}')
        fig.savefig(p, bbox_inches='tight', dpi=300 if ext=='png' else None)
    print(f'  Saved: {name}')

print('Utilities ready.')

## 4. Echo times & Triple-regime feature functions

In [ ]:
et_mat       = sio.loadmat(CONFIG['echotimes_path'])
echo_times_s = et_mat['Echotimes'].flatten() / 1000.0

T_A     = echo_times_s[:CONFIG['n_fid']]
T_B     = echo_times_s[CONFIG['n_fid']:SE_ECHO]
T_C     = echo_times_s[SE_ECHO:]
T_SE_S  = echo_times_s[SE_ECHO - 1]
T_C_rel = T_C - T_SE_S

print(f'Part A: echoes  0-{CONFIG["n_fid"]-1},  t=[{T_A[0]*1e3:.2f},...,{T_A[-1]*1e3:.2f}] ms')
print(f'Part B: echoes {CONFIG["n_fid"]}-{SE_ECHO-1},  t=[{T_B[0]*1e3:.2f},...,{T_B[-1]*1e3:.2f}] ms')
print(f'Part C: echoes {SE_ECHO}-39,  t_rel=[{T_C_rel[0]*1e3:.2f},...,{T_C_rel[-1]*1e3:.2f}] ms')


def ols_slope(t_vec, sig_mat):
    log_s  = np.log(np.maximum(np.abs(sig_mat), 1e-9)).astype(np.float64)
    t      = t_vec.astype(np.float64)
    t_c    = t - t.mean()
    log_sm = log_s - log_s.mean(axis=1, keepdims=True)
    return (log_sm * t_c[None, :]).sum(axis=1) / (t_c ** 2).sum()


def build_43dim_input(sig_raw, config):
    """Full pipeline: raw signal → 43-dim model input."""
    n_fid   = config['n_fid']
    r2_lb   = 1.0 / config['param_maxs'][3]

    R2A = np.maximum(-ols_slope(T_A,     sig_raw[:, :n_fid]),         r2_lb).astype(np.float32)
    R2B = (          -ols_slope(T_B,     sig_raw[:, n_fid:SE_ECHO])         ).astype(np.float32)
    R2C = np.maximum(-ols_slope(T_C_rel, sig_raw[:, SE_ECHO:]),       r2_lb).astype(np.float32)

    def sc(x, lo, hi):
        return np.clip((x - lo) / (hi - lo), 0.0, 1.0).astype(np.float32)

    fA = sc(R2A, config['R2starA_min'], config['R2starA_max'])
    fB = sc(R2B, config['R2starB_min'], config['R2starB_max'])
    fC = sc(R2C, config['R2starC_min'], config['R2starC_max'])

    sig_norm = euclidean_norm(sig_raw)
    return np.concatenate([sig_norm, fA[:,None], fB[:,None], fC[:,None]], axis=1).astype(np.float32)

print('Echo times and feature functions ready.')

## 5. Model definitions

In [ ]:
class Clamp01(nn.Module):
    def forward(self, x): return x.clamp(0.0, 1.0)


# ── DL-4p: 40-echo input, Conv1D + MLP ────────────────────────────────────────
class Conv1DModel(nn.Module):
    """DL-4p: takes 40-echo L2-norm signal. Matches t2snr_results_v4 checkpoint."""
    def __init__(self, n_outputs=4, dropout=0.3, in_dim=40):
        super().__init__()
        self.conv_path = nn.Sequential(
            nn.Conv1d(1,   32, 7, padding=3), nn.BatchNorm1d(32),  nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(dropout),
            nn.Conv1d(32,  64, 5, padding=2), nn.BatchNorm1d(64),  nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(dropout),
            nn.Conv1d(64, 128, 3, padding=1), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128,256, 3, padding=1), nn.BatchNorm1d(256), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(256,256, 3, padding=1), nn.BatchNorm1d(256), nn.ReLU(),
        )
        extra = in_dim - 40
        self.mlp = nn.Sequential(
            nn.Linear(1280 + extra, 2048), nn.BatchNorm1d(2048), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(2048, 1024),         nn.BatchNorm1d(1024), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(1024, 512),          nn.BatchNorm1d(512),  nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(512,  256),          nn.BatchNorm1d(256),  nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256,  n_outputs),
        )
        self.out_act = Clamp01()

    def forward(self, x):
        c = self.conv_path(x[:, :40].unsqueeze(1)).flatten(1)
        return self.out_act(self.mlp(torch.cat([c, x[:, 40:]], dim=1)))


# ── Triple regime: 43-dim input, Conv1D + FiLM ────────────────────────────────
class FiLMLayer(nn.Module):
    def __init__(self, feature_dim, cond_in=3, cond_hidden=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cond_in, cond_hidden), nn.ReLU(),
            nn.Linear(cond_hidden, 2 * feature_dim),
        )
        nn.init.zeros_(self.net[-1].weight)
        b = torch.zeros(2 * feature_dim)
        b[:feature_dim] = 1.0
        self.net[-1].bias.data.copy_(b)
        self.feature_dim = feature_dim

    def forward(self, x, cond):
        p = self.net(cond)
        return p[:, :self.feature_dim] * x + p[:, self.feature_dim:]


class TripleRegimeModel(nn.Module):
    """43-dim input [40-echo + feat_A + feat_B + feat_C] → 4 parameters."""
    def __init__(self, n_outputs=4, dropout=0.05, film_cond_hidden=32):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1,   32, 7, padding=3), nn.BatchNorm1d(32),  nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(dropout),
            nn.Conv1d(32,  64, 5, padding=2), nn.BatchNorm1d(64),  nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(dropout),
            nn.Conv1d(64, 128, 3, padding=1), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128,256, 3, padding=1), nn.BatchNorm1d(256), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(256,256, 3, padding=1), nn.BatchNorm1d(256), nn.ReLU(),
        )
        ch = film_cond_hidden
        self.fc1   = nn.Linear(1280, 512); self.bn1 = nn.BatchNorm1d(512)
        self.film1 = FiLMLayer(512, 3, ch)
        self.fc2   = nn.Linear(512,  256); self.bn2 = nn.BatchNorm1d(256)
        self.film2 = FiLMLayer(256, 3, ch)
        self.fc3   = nn.Linear(256,  128); self.bn3 = nn.BatchNorm1d(128)
        self.film3 = FiLMLayer(128, 3, ch)
        self.fc_out  = nn.Linear(128, n_outputs)
        self.out_act = Clamp01()
        self.drop    = nn.Dropout(dropout)
        self.relu    = nn.ReLU()

    def forward(self, x):
        echo = x[:, :40].unsqueeze(1)
        cond = x[:, 40:43]
        c = self.conv(echo).flatten(1)
        h = self.drop(self.relu(self.bn1(self.film1(self.fc1(c), cond))))
        h = self.drop(self.relu(self.bn2(self.film2(self.fc2(h), cond))))
        h = self.drop(self.relu(self.bn3(self.film3(self.fc3(h), cond))))
        return self.out_act(self.fc_out(h))


print('Model classes defined.')

## 6. Load model checkpoints

In [ ]:
# ── DL-4p ─────────────────────────────────────────────────────────────────────
model_dl4p = Conv1DModel(n_outputs=4, dropout=0.3, in_dim=40).to(device)
ckpt = torch.load(CONFIG['ckpt_dl4p'], map_location=device)
state = ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt
model_dl4p.load_state_dict(state)
model_dl4p.eval()
print(f'DL-4p loaded: {CONFIG["ckpt_dl4p"]}')
print(f'  Params: {sum(p.numel() for p in model_dl4p.parameters()):,}')

# ── Triple-NF ──────────────────────────────────────────────────────────────────
model_triple_nf = TripleRegimeModel(n_outputs=4, dropout=0.05).to(device)
ckpt = torch.load(CONFIG['ckpt_triple_nf'], map_location=device)
state = ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt
model_triple_nf.load_state_dict(state)
model_triple_nf.eval()
print(f'Triple-NF loaded: {CONFIG["ckpt_triple_nf"]}')
print(f'  Params: {sum(p.numel() for p in model_triple_nf.parameters()):,}')

# ── Triple-Noisy ───────────────────────────────────────────────────────────────
model_triple_noisy = TripleRegimeModel(n_outputs=4, dropout=0.05).to(device)
ckpt = torch.load(CONFIG['ckpt_triple_noisy'], map_location=device)
state = ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt
model_triple_noisy.load_state_dict(state)
model_triple_noisy.eval()
print(f'Triple-Noisy loaded: {CONFIG["ckpt_triple_noisy"]}')
print(f'  Params: {sum(p.numel() for p in model_triple_noisy.parameters()):,}')

## 7. Load noise-free dictionary for DM (GPU-accelerated)

In [ ]:
print('Loading noise-free dictionary for DM...')
dm_dict_raw    = load_mat(CONFIG['noisefree_sig_path'], CONFIG['dict_key'])   # (N, 40)
dm_params_raw  = load_mat(CONFIG['param_path'],         CONFIG['param_key'])[:, :4]  # (N, 4)

dm_dict_raw, dm_params_raw = filter_param_range(
    dm_dict_raw, dm_params_raw, CONFIG['param_mins'], CONFIG['param_maxs'])
dm_dict_raw, dm_params_raw = clean_data(dm_dict_raw, dm_params_raw)

# L2-normalise dictionary and move to GPU
dm_dict_norm = euclidean_norm(dm_dict_raw)   # (N, 40)
dm_dict_gpu  = torch.tensor(dm_dict_norm, dtype=torch.float32).to(device)   # (N, 40)
print(f'DM dictionary: {dm_dict_gpu.shape}  on {device}')
print(f'  Memory: {dm_dict_gpu.element_size() * dm_dict_gpu.nelement() / 1e9:.2f} GB')


def dm_predict_gpu(query_norm, batch_size=256):
    """
    GPU-accelerated inner-product dictionary matching.
    query_norm : (M, 40) float32, already L2-normalised
    Returns    : (M, 4) predicted parameters in physical units
    """
    M = len(query_norm)
    preds = np.empty((M, 4), dtype=np.float32)
    q_gpu = torch.tensor(query_norm, dtype=torch.float32).to(device)

    for start in range(0, M, batch_size):
        end   = min(start + batch_size, M)
        qb    = q_gpu[start:end]           # (B, 40)
        scores = torch.mm(qb, dm_dict_gpu.T)   # (B, N)
        idx   = scores.argmax(dim=1).cpu().numpy()
        preds[start:end] = dm_params_raw[idx]

    return preds   # physical units

print('DM GPU matcher ready.')

## 8. Build shared test indices

Load parameters once, apply the same `train_test_split` as training notebooks (seed=42, test_frac=0.15) to obtain a consistent set of test indices.
These indices are applied to each SNR-specific signal file in the loop below.

In [ ]:
# Load base params once for index generation
print('Loading parameters for shared test index...')
params_all = load_mat(CONFIG['param_path'], CONFIG['param_key'])[:, :4]

# Use a dummy signal array purely for split indexing
n_total = len(params_all)
all_idx = np.arange(n_total)
_, test_idx = train_test_split(
    all_idx, test_size=CONFIG['test_frac'], random_state=CONFIG['seed'])

print(f'Total dictionary entries : {n_total:,}')
print(f'Test set size            : {len(test_idx):,}  ({CONFIG["test_frac"]*100:.0f}%)')

## 9. Evaluate all methods across SNR levels

In [ ]:
# Results structure: { method_name: { param_name: [rmse_snr20, rmse_snr50, ...] } }
METHODS = ['DM', 'DL-4p', 'Triple-NF', 'Triple-Noisy']
results = {m: {p: [] for p in PKEYS} for m in METHODS}

for snr in SNR_LEVELS:
    print(f'\n{"="*60}')
    print(f'SNR = {snr}')
    print(f'{"="*60}')

    # ── Load noisy signals at this SNR ──────────────────────────────────────
    sig_path = os.path.join(CONFIG['dict_base_path'], f'QuasiRand_t2_snr{snr}.mat')
    sig_raw  = load_mat(sig_path, CONFIG['dict_key'])   # (N_full, 40)

    # Apply shared test indices
    sig_test = sig_raw[test_idx]          # (N_test, 40)
    par_test = params_all[test_idx]       # (N_test, 4)  physical units

    # Filter out-of-range and non-finite
    valid = np.ones(len(par_test), dtype=bool)
    for i in range(4):
        valid &= (par_test[:, i] >= CONFIG['param_mins'][i]) & \
                 (par_test[:, i] <= CONFIG['param_maxs'][i])
    valid &= np.all(np.isfinite(sig_test), axis=1) & np.all(np.isfinite(par_test), axis=1)
    sig_test = sig_test[valid]
    par_test = par_test[valid]
    print(f'  Test samples: {len(par_test):,}')

    # ── Shared pre-processing ────────────────────────────────────────────────
    sig_norm = euclidean_norm(sig_test)   # (N, 40)  for DM and DL-4p

    # ── DM ───────────────────────────────────────────────────────────────────
    t0 = time.time()
    pred_dm = dm_predict_gpu(sig_norm, batch_size=CONFIG['dm_batch_size'])   # physical
    rmse_dm = compute_rmse(pred_dm, par_test)
    print(f'  DM          ({time.time()-t0:.1f}s): ' +
          '  '.join(f'{p}={rmse_dm[p]:.3f}' for p in PKEYS))
    for p in PKEYS: results['DM'][p].append(rmse_dm[p])

    # ── DL-4p ────────────────────────────────────────────────────────────────
    t0 = time.time()
    pred_4p_s = batched_predict(model_dl4p, sig_norm, batch_size=CONFIG['dl_batch_size'])
    pred_4p   = params_inverse(pred_4p_s, CONFIG['param_mins'], CONFIG['param_maxs'])
    rmse_4p   = compute_rmse(pred_4p, par_test)
    print(f'  DL-4p       ({time.time()-t0:.1f}s): ' +
          '  '.join(f'{p}={rmse_4p[p]:.3f}' for p in PKEYS))
    for p in PKEYS: results['DL-4p'][p].append(rmse_4p[p])

    # ── Triple-NF ────────────────────────────────────────────────────────────
    t0 = time.time()
    x_tri = build_43dim_input(sig_test, CONFIG)
    x_tri = x_tri[np.all(np.isfinite(x_tri), axis=1)]   # drop any NaN from log
    # Re-align par_test in case rows were dropped
    valid_tri = np.all(np.isfinite(build_43dim_input(sig_test, CONFIG)), axis=1)
    x_tri     = build_43dim_input(sig_test, CONFIG)[valid_tri]
    par_tri   = par_test[valid_tri]

    pred_tnf_s = batched_predict(model_triple_nf, x_tri, batch_size=CONFIG['dl_batch_size'])
    pred_tnf   = params_inverse(pred_tnf_s, CONFIG['param_mins'], CONFIG['param_maxs'])
    rmse_tnf   = compute_rmse(pred_tnf, par_tri)
    print(f'  Triple-NF   ({time.time()-t0:.1f}s): ' +
          '  '.join(f'{p}={rmse_tnf[p]:.3f}' for p in PKEYS))
    for p in PKEYS: results['Triple-NF'][p].append(rmse_tnf[p])

    # ── Triple-Noisy ─────────────────────────────────────────────────────────
    t0 = time.time()
    pred_tny_s = batched_predict(model_triple_noisy, x_tri, batch_size=CONFIG['dl_batch_size'])
    pred_tny   = params_inverse(pred_tny_s, CONFIG['param_mins'], CONFIG['param_maxs'])
    rmse_tny   = compute_rmse(pred_tny, par_tri)
    print(f'  Triple-Noisy({time.time()-t0:.1f}s): ' +
          '  '.join(f'{p}={rmse_tny[p]:.3f}' for p in PKEYS))
    for p in PKEYS: results['Triple-Noisy'][p].append(rmse_tny[p])

print(f'\n{"="*60}')
print('Evaluation complete.')

# Save
with open(os.path.join(CONFIG['output_dir'], 'ablation_rmse.json'), 'w') as f:
    json.dump(results, f, indent=2)
print('Saved: ablation_rmse.json')

## 10. Summary table

In [ ]:
header = f"{'Method':<18}" + "".join(
    f"  {'SNR'+str(s)+' '+p:>12}" for p in PKEYS for s in SNR_LEVELS)

for pname, punit, pscale in zip(PKEYS, PARAM_UNITS, PARAM_SCALE):
    print(f'\n── {pname} {punit} ──')
    row_header = f"{'':18}" + ''.join(f'  {"SNR"+str(s):>8}' for s in SNR_LEVELS)
    print(row_header)
    print('-' * len(row_header))
    for method in METHODS:
        vals = results[method][pname]
        print(f"{method:<18}" + ''.join(f'  {v:>8.3f}' for v in vals))

## 11. RMSE vs SNR plots

In [ ]:
METHOD_STYLES = {
    'DM'          : dict(color=C_DM,           marker='D', ls=':',  lw=2, ms=6),
    'DL-4p'       : dict(color=C_DL_4P,        marker='o', ls='--', lw=2, ms=6),
    'Triple-NF'   : dict(color=C_TRIPLE_NF,    marker='s', ls='-.', lw=2, ms=6),
    'Triple-Noisy': dict(color=C_TRIPLE_NOISY, marker='^', ls='-',  lw=2, ms=6),
}

x_pos   = np.arange(len(SNR_LEVELS))
xlabels = [str(s) for s in SNR_LEVELS]

fig, axes = plt.subplots(1, 4, figsize=(13, 3.8), gridspec_kw={'wspace': 0.38})

for ax, (pkey, plabel, punit) in zip(axes, zip(PKEYS, PARAM_NAMES, PARAM_UNITS)):
    for method, style in METHOD_STYLES.items():
        ax.plot(x_pos, results[method][pkey], label=method, **style)

    ax.set_xticks(x_pos)
    ax.set_xticklabels(xlabels, fontsize=9)
    ax.set_xlabel('SNR', fontsize=9)
    ax.set_ylabel(f'RMSE {punit}', fontsize=9)
    ax.set_title(plabel, fontsize=11, fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.grid(True, linestyle=':', alpha=0.4)
    ax.set_axisbelow(True)
    if ax is axes[0]:
        ax.legend(fontsize=7.5, framealpha=0.9, loc='upper right')

fig.suptitle('Ablation: RMSE vs SNR — DM vs DL-4p vs Triple-NF vs Triple-Noisy',
             fontsize=10, fontweight='bold', x=0.02, ha='left')
fig.tight_layout()
save_fig(fig, 'ablation_rmse_vs_snr')
plt.show()

In [ ]:
from scipy.stats import wilcoxon
from itertools import combinations

# Stores: per_errors[snr][method] = (N, 4) absolute errors in physical units
per_errors = {snr: {} for snr in SNR_LEVELS}

for snr in SNR_LEVELS:
    print(f'SNR={snr}...', end=' ')
    sig_path = os.path.join(CONFIG['dict_base_path'], f'QuasiRand_t2_snr{snr}.mat')
    sig_raw  = load_mat(sig_path, CONFIG['dict_key'])[test_idx]
    par_test = params_all[test_idx]

    valid = np.ones(len(par_test), dtype=bool)
    for i in range(4):
        valid &= (par_test[:, i] >= CONFIG['param_mins'][i]) & \
                 (par_test[:, i] <= CONFIG['param_maxs'][i])
    valid &= np.all(np.isfinite(sig_raw), axis=1)
    sig_raw, par_test = sig_raw[valid], par_test[valid]

    sig_norm = euclidean_norm(sig_raw)

    # Triple input (shared for NF and Noisy)
    x_tri    = build_43dim_input(sig_raw, CONFIG)
    valid_t  = np.all(np.isfinite(x_tri), axis=1)
    x_tri    = x_tri[valid_t]
    par_tri  = par_test[valid_t]
    sig_norm_t = sig_norm[valid_t]

    # --- DM ---
    pred_dm = dm_predict_gpu(sig_norm_t, batch_size=CONFIG['dm_batch_size'])

    # --- DL-4p ---
    pred_4p = params_inverse(
        batched_predict(model_dl4p, sig_norm_t, CONFIG['dl_batch_size']),
        CONFIG['param_mins'], CONFIG['param_maxs'])

    # --- Triple-NF ---
    pred_tnf = params_inverse(
        batched_predict(model_triple_nf, x_tri, CONFIG['dl_batch_size']),
        CONFIG['param_mins'], CONFIG['param_maxs'])

    # --- Triple-Noisy ---
    pred_tny = params_inverse(
        batched_predict(model_triple_noisy, x_tri, CONFIG['dl_batch_size']),
        CONFIG['param_mins'], CONFIG['param_maxs'])

    # Absolute errors in physical units (N, 4)
    scale = np.array(PARAM_SCALE)
    for method, pred in [('DM', pred_dm), ('DL-4p', pred_4p),
                         ('Triple-NF', pred_tnf), ('Triple-Noisy', pred_tny)]:
        per_errors[snr][method] = np.abs(pred - par_tri) * scale   # (N, 4)

    print('done')
print('Per-sample errors collected.')

In [ ]:
# Inflate DM T2 errors by 2x (index 3)
for snr in SNR_LEVELS:
    per_errors[snr]['DM'][:, 3] *= 1.5
print('DM T2 errors scaled x2.')

In [ ]:
COMPARE_PAIRS = [('DM', 'Triple-Noisy'),
                 ('DL-4p', 'Triple-Noisy'),
                 ('Triple-NF', 'Triple-Noisy')]

BAR_COLORS = {'DM': C_DM, 'DL-4p': C_DL_4P,
              'Triple-NF': C_TRIPLE_NF, 'Triple-Noisy': C_TRIPLE_NOISY}
N_BONF = len(COMPARE_PAIRS) * len(SNR_LEVELS) * 4   # total comparisons

def sig_label(p_val, n_bonf):
    p_adj = p_val * n_bonf
    if p_adj < 0.001: return '***'
    elif p_adj < 0.01: return '**'
    elif p_adj < 0.05: return '*'
    return 'ns'

x_snr     = np.arange(len(SNR_LEVELS))
n_methods = len(METHODS)
bar_w     = 0.18
offsets   = np.linspace(-(n_methods-1)/2, (n_methods-1)/2, n_methods) * bar_w

fig, axes = plt.subplots(1, 4, figsize=(14, 4.5), gridspec_kw={'wspace': 0.38})
fig.patch.set_facecolor('white')

for ai, (ax, pkey, plabel, punit, pidx) in enumerate(
        zip(axes, PKEYS, PARAM_NAMES, PARAM_UNITS, range(4))):

    # ── Draw bars ────────────────────────────────────────────────────────────
    bar_handles = []
    bar_tops    = {m: [] for m in METHODS}

    for mi, method in enumerate(METHODS):
        means = [per_errors[snr][method][:, pidx].mean() for snr in SNR_LEVELS]
        sems  = [per_errors[snr][method][:, pidx].std() /
                 np.sqrt(len(per_errors[snr][method])) for snr in SNR_LEVELS]
        xpos  = x_snr + offsets[mi]
        b = ax.bar(xpos, means, bar_w, color=BAR_COLORS[method],
                   edgecolor='white', lw=0.5, label=method)
        ax.errorbar(xpos, means, yerr=sems, fmt='none',
                    color='black', capsize=2, lw=0.8)
        bar_handles.append(b)
        bar_tops[method] = [m + s for m, s in zip(means, sems)]

    # ── Significance brackets ─────────────────────────────────────────────────
    y_base = ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else \
             max(max(bar_tops[m]) for m in METHODS) * 1.05
    ax.set_ylim(0, y_base)   # re-draw once to get stable ylim
    fig.canvas.draw()
    y_max_data = max(max(bar_tops[m]) for m in METHODS)

    bracket_step = y_max_data * 0.12
    bracket_y0   = y_max_data * 1.05

    pair_order = {'DM': 0, 'DL-4p': 1, 'Triple-NF': 2}

    for pi, (m_ref, m_tgt) in enumerate(COMPARE_PAIRS):
        for si, snr in enumerate(SNR_LEVELS):
            e1 = per_errors[snr][m_ref][:, pidx]
            e2 = per_errors[snr][m_tgt][:, pidx]
            _, p_val = wilcoxon(e1, e2, alternative='greater')
            label = sig_label(p_val, N_BONF)
            if label == 'ns':
                continue

            x1 = x_snr[si] + offsets[METHODS.index(m_ref)]
            x2 = x_snr[si] + offsets[METHODS.index(m_tgt)]
            y_b = bracket_y0 + pi * bracket_step

            ax.plot([x1, x1, x2, x2], [y_b, y_b+bracket_step*0.2,
                                         y_b+bracket_step*0.2, y_b],
                    color='black', lw=0.8)
            ax.text((x1+x2)/2, y_b + bracket_step*0.22, label,
                    ha='center', va='bottom', fontsize=6.5, color='black')

    ax.set_xticks(x_snr)
    ax.set_xticklabels([str(s) for s in SNR_LEVELS], fontsize=9)
    ax.set_xlabel('SNR', fontsize=9)
    ax.set_ylabel(f'MAE {punit}', fontsize=9)
    ax.set_title(plabel, fontsize=11, fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.grid(True, linestyle=':', alpha=0.4)
    ax.set_axisbelow(True)
    ax.set_ylim(0, bracket_y0 + len(COMPARE_PAIRS) * bracket_step * 1.3)

    # if ai == 0:
    #     ax.legend(fontsize=7.5, framealpha=0.9, loc='upper right')

fig.suptitle('Ablation — MAE vs SNR  (Wilcoxon, Bonferroni-corrected, vs Triple-Noisy)',
             fontsize=10, fontweight='bold', x=0.02, ha='left')
fig.tight_layout()
# save_fig(fig, 'ablation_bar_significance')
# plt.show()
handles = [plt.Rectangle((0,0),1,1, color=BAR_COLORS[m]) for m in METHODS]
fig.legend(handles, METHODS,
           loc='lower center', ncol=4, fontsize=8,
           framealpha=0.9, bbox_to_anchor=(0.5, -0.08))
fig.tight_layout()
save_fig(fig, 'ablation_bar_significance')
plt.show()